In [1]:
from pathlib import Path
import json
import re
import hashlib
import statistics
from datetime import datetime, timezone
from collections import Counter, defaultdict

PROJECT_ROOT = Path(r"D:\Project_Github\gold assisstant")

INPUTS = {
    "thai_pbs_gold": [
        PROJECT_ROOT / "dataset" / "news" / "pbs" / "gold_audit_direct.jsonl",
    ],
    "fed_press_releases": [
        PROJECT_ROOT / "dataset" / "news" / "fed" /
        "fed_press_releases_cleaned_audit" /
        "fed_press_releases_classified.jsonl",
    ],
    "bls_cpi": [
        PROJECT_ROOT / "dataset" / "news" / "bls" / "cpi" / "clean",
    ],
    "bls_nfp": [
        PROJECT_ROOT / "dataset" / "news" / "bls" / "nfp" / "clean",
    ],
}

OUTPUT_DIR = (
    PROJECT_ROOT / "embedding" / "artifacts" / "chunking_real" / "structure_aware"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chunk size ใช้หน่วยตัวอักษร ไม่ใช่ tokenizer-exact tokens
TARGET_CHARS = 1200
MAX_CHARS = 1800
MIN_CHARS = 250
OVERLAP_CHARS = 120

SCHEMA_VERSION = "structure_aware_real_v1"

print("Project:", PROJECT_ROOT)
print("Output:", OUTPUT_DIR)

for source, paths in INPUTS.items():
    for path in paths:
        print(f"{source:22s} | exists={path.exists()} | {path}")

Project: D:\Project_Github\gold assisstant
Output: D:\Project_Github\gold assisstant\embedding\artifacts\chunking_real\structure_aware
thai_pbs_gold          | exists=True | D:\Project_Github\gold assisstant\dataset\news\pbs\gold_audit_direct.jsonl
fed_press_releases     | exists=True | D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned_audit\fed_press_releases_classified.jsonl
bls_cpi                | exists=True | D:\Project_Github\gold assisstant\dataset\news\bls\cpi\clean
bls_nfp                | exists=True | D:\Project_Github\gold assisstant\dataset\news\bls\nfp\clean


In [2]:
EXCLUDED_FILENAME_PARTS = {
    "manifest",
    "metadata",
    "index",
    "chunk",
    "embedding",
    "vector",
    "coverage",
    "validation",
    "ground_truth",
}

ALLOWED_SUFFIXES = {".json", ".jsonl", ".ndjson"}


def read_jsonl(path):
    with path.open("r", encoding="utf-8-sig") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"[WARN] {path.name}:{line_no}: {e}")
                continue

            if isinstance(obj, dict):
                yield obj


def read_json(path):
    try:
        obj = json.loads(path.read_text(encoding="utf-8-sig"))
    except Exception as e:
        print(f"[WARN] อ่านไม่ได้: {path} | {e}")
        return

    if isinstance(obj, dict):
        for key in ("documents", "articles", "reports",
                    "items", "data", "records"):
            if isinstance(obj.get(key), list):
                for item in obj[key]:
                    if isinstance(item, dict):
                        yield item
                return

        yield obj

    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict):
                yield item


def iter_records(paths):
    for input_path in paths:
        if not input_path.exists():
            print(f"[WARN] ไม่พบ Path: {input_path}")
            continue

        if input_path.is_file():
            if input_path.suffix.lower() in {".jsonl", ".ndjson"}:
                yield from (
                    (input_path, row)
                    for row in read_jsonl(input_path)
                )
            elif input_path.suffix.lower() == ".json":
                yield from (
                    (input_path, row)
                    for row in read_json(input_path)
                )
            continue

        files = sorted(
            p for p in input_path.rglob("*")
            if p.is_file()
            and p.suffix.lower() in ALLOWED_SUFFIXES
            and not any(
                part in p.name.lower()
                for part in EXCLUDED_FILENAME_PARTS
            )
            and not any(
                folder in part.lower()
                for part in p.parts
                for folder in ("embedding", "vector", "chunk")
            )
        )

        for path in files:
            if path.suffix.lower() in {".jsonl", ".ndjson"}:
                yield from (
                    (path, row)
                    for row in read_jsonl(path)
                )
            else:
                yield from (
                    (path, row)
                    for row in read_json(path)
                )


print("Loader functions ready.")

Loader functions ready.


In [3]:
def get_text(record, *keys):
    for key in keys:
        value = record.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return ""


def clean_text(value):
    if value is None:
        return ""

    if isinstance(value, list):
        return "\n".join(
            clean_text(x) for x in value if x is not None
        )

    if isinstance(value, dict):
        # รักษาโครงสร้าง key-value เช่น ตัวเลขและหน่วย
        lines = []

        for key, val in value.items():
            val_text = clean_text(val)

            if val_text:
                lines.append(f"{key}: {val_text}")

        return "\n".join(lines)

    text = str(value)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def split_body_into_sections(body):
    body = clean_text(body)

    if not body:
        return []

    sections = []
    current_heading = "Body"
    buffer = []

    def flush():
        nonlocal buffer

        content = "\n".join(buffer).strip()

        if content:
            sections.append({
                "heading": current_heading,
                "text": content,
            })

        buffer = []

    for line in body.splitlines():
        line = line.strip()

        if not line:
            flush()
            continue

        is_heading = (
            bool(re.match(r"^#{1,6}\s+\S", line))
            or (
                len(line) <= 100
                and len(line) >= 4
                and line.isupper()
                and any(ch.isalpha() for ch in line)
            )
        )

        if is_heading:
            flush()
            current_heading = re.sub(r"^#{1,6}\s*", "", line)
        else:
            buffer.append(line)

    flush()

    if not sections and body:
        sections = [{"heading": "Body", "text": body}]

    return sections


def make_stable_id(source, original_id, title, content):
    identity = (
        f"{source}|{original_id}|{title}|{content[:500]}"
    )

    return hashlib.sha1(
        identity.encode("utf-8")
    ).hexdigest()[:16]


def normalize_record(source, path, record):
    title = get_text(record, "title", "headline", "name")
    abstract = get_text(
        record, "abstract", "summary", "description", "dek"
    )
    publication_date = get_text(
        record, "publication_date", "published_at",
        "published", "date", "pub_date"
    )
    report_period = get_text(
        record, "report_period", "period", "reference_period"
    )
    url = get_text(record, "url", "source_url", "link")

    original_id = get_text(
        record, "document_id", "doc_id", "article_id", "id"
    )

    sections = []

    # รองรับ BLS หรือข้อมูลที่เก็บส่วนของรายงานเป็น chunks[]
    raw_sections = record.get("chunks")

    if isinstance(raw_sections, list):
        for i, item in enumerate(raw_sections):
            if not isinstance(item, dict):
                continue

            heading = get_text(
                item, "heading", "title", "section", "name"
            ) or f"Section {i + 1}"

            content = clean_text(
                item.get("content")
                or item.get("text")
                or item.get("body")
            )

            if content:
                sections.append({
                    "heading": heading,
                    "text": content,
                })

    # รองรับ sections[] และ content_sections[]
    if not sections:
        for key in ("sections", "content_sections"):
            raw = record.get(key)

            if not isinstance(raw, list):
                continue

            for i, item in enumerate(raw):
                if isinstance(item, dict):
                    heading = get_text(
                        item, "heading", "title", "name"
                    ) or f"Section {i + 1}"

                    content = clean_text(
                        item.get("content")
                        or item.get("text")
                        or item.get("body")
                    )
                else:
                    heading = f"Section {i + 1}"
                    content = clean_text(item)

                if content:
                    sections.append({
                        "heading": heading,
                        "text": content,
                    })

    # ถ้าไม่มีโครงสร้าง sections ให้ใช้ body ต้นฉบับ
    body = get_text(
        record, "content", "text", "body",
        "full_text", "article_text"
    )

    if not sections and body:
        sections = split_body_into_sections(body)

    # เพิ่ม abstract เฉพาะเมื่อไม่ได้เป็นส่วนหนึ่งอยู่แล้ว
    if abstract and not any(
        s["heading"].strip().lower() == "abstract"
        for s in sections
    ):
        sections.insert(0, {
            "heading": "Abstract",
            "text": abstract,
        })

    if not title and not sections:
        return None

    full_content = "\n".join(
        section["text"] for section in sections
    )

    if not full_content.strip():
        return None

    document_id = make_stable_id(
        source, original_id, title, full_content
    )

    return {
        "source": source,
        "document_id": document_id,
        "original_id": original_id,
        "title": title or "(Untitled)",
        "publication_date": publication_date,
        "report_period": report_period,
        "url": url,
        "language": "th" if source == "thai_pbs_gold" else "en",
        "source_file": str(path),
        "source_classification": record.get("classification"),
        "sections": sections,
    }


print("Normalization functions ready.")

Normalization functions ready.


In [4]:
documents = []
source_counts = Counter()
records_seen = Counter()
errors = []

seen_fingerprints = set()

for source, paths in INPUTS.items():
    for path, record in iter_records(paths):
        records_seen[source] += 1

        try:
            doc = normalize_record(source, path, record)

            if doc is None:
                continue

            # Deduplicate เฉพาะเนื้อหาที่เหมือนกันภายใน source เดียวกัน
            fingerprint_text = (
                doc["source"] + "|"
                + doc["title"] + "|"
                + doc["publication_date"] + "|"
                + "\n".join(
                    s["text"] for s in doc["sections"]
                )
            )

            fingerprint = hashlib.sha1(
                fingerprint_text.encode("utf-8")
            ).hexdigest()

            if fingerprint in seen_fingerprints:
                continue

            seen_fingerprints.add(fingerprint)
            documents.append(doc)
            source_counts[source] += 1

        except Exception as exc:
            errors.append({
                "source": source,
                "path": str(path),
                "error": repr(exc),
            })

print("=== INGESTION SUMMARY ===")

for source in INPUTS:
    print(
        f"{source:22s} "
        f"records_read={records_seen[source]:,} "
        f"documents_kept={source_counts[source]:,}"
    )

print("Total documents:", f"{len(documents):,}")
print("Errors:", len(errors))

if errors:
    print("\nFirst 10 errors:")
    for error in errors[:10]:
        print(error)

if not documents:
    raise RuntimeError(
        "ไม่พบเอกสารที่นำมาทำ Chunk ได้ "
        "โปรดตรวจสอบ Path และ Schema ของไฟล์ต้นทาง"
    )

=== INGESTION SUMMARY ===
thai_pbs_gold          records_read=9 documents_kept=9
fed_press_releases     records_read=4,553 documents_kept=4,553
bls_cpi                records_read=248 documents_kept=248
bls_nfp                records_read=248 documents_kept=248
Total documents: 5,058
Errors: 0


In [5]:
def pack_short_sections(sections):
    packed = []

    for section in sections:
        heading = clean_text(section.get("heading")) or "Body"
        content = clean_text(section.get("text"))

        if not content:
            continue

        # รวมเฉพาะ section สั้นที่อยู่ติดกันและยังไม่เกิน target
        if (
            packed
            and len(packed[-1]["text"]) < MIN_CHARS
            and len(packed[-1]["text"]) + len(content) + 80
                <= TARGET_CHARS
        ):
            packed[-1]["heading"] += " / " + heading
            packed[-1]["text"] += "\n\n" + content
        else:
            packed.append({
                "heading": heading,
                "text": content,
            })

    return packed


def split_long_text(text, max_chars=MAX_CHARS):
    text = clean_text(text)

    if not text:
        return []

    if len(text) <= max_chars:
        return [text]

    # ให้ความสำคัญกับ paragraph boundaries ก่อน
    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", text)
        if p.strip()
    ]

    if not paragraphs:
        paragraphs = [text]

    chunks = []
    current = ""

    def flush():
        nonlocal current

        if current.strip():
            chunks.append(current.strip())

        current = ""

    for paragraph in paragraphs:
        # กรณีย่อหน้าเดียวมีขนาดเกินขีดจำกัด
        if len(paragraph) > max_chars:
            flush()

            # ลองแบ่งตามประโยคก่อน
            sentences = re.split(
                r"(?<=[.!?。！？])\s+",
                paragraph
            )

            sentence_buffer = ""

            for sentence in sentences:
                sentence = sentence.strip()

                if not sentence:
                    continue

                if len(sentence) > max_chars:
                    if sentence_buffer:
                        chunks.append(sentence_buffer.strip())
                        sentence_buffer = ""

                    # Hard split เป็นทางเลือกสุดท้าย
                    start = 0

                    while start < len(sentence):
                        end = min(start + max_chars, len(sentence))

                        # หาจุดตัด whitespace ใกล้ปลาย chunk
                        if end < len(sentence):
                            boundary = sentence.rfind(
                                " ", start, end
                            )

                            if boundary > start + max_chars * 0.6:
                                end = boundary

                        piece = sentence[start:end].strip()

                        if piece:
                            chunks.append(piece)

                        start = end

                else:
                    candidate = (
                        sentence_buffer + " " + sentence
                    ).strip()

                    if len(candidate) <= max_chars:
                        sentence_buffer = candidate
                    else:
                        if sentence_buffer:
                            chunks.append(sentence_buffer.strip())

                        sentence_buffer = sentence

            if sentence_buffer:
                chunks.append(sentence_buffer.strip())

            continue

        candidate = (
            current + "\n\n" + paragraph
        ).strip()

        if len(candidate) <= max_chars:
            current = candidate
        else:
            flush()
            current = paragraph

    flush()

    return chunks


def add_overlap(chunks, overlap_chars=OVERLAP_CHARS):
    """
    เพิ่ม overlap จากท้าย chunk ก่อนหน้า
    เฉพาะเมื่อ chunk ถัดไปไม่เกิน MAX_CHARS
    """
    if not chunks:
        return []

    result = [chunks[0]]

    for chunk in chunks[1:]:
        previous_tail = result[-1][-overlap_chars:].strip()

        candidate = (
            previous_tail + "\n" + chunk
        ).strip() if previous_tail else chunk

        if len(candidate) <= MAX_CHARS:
            result.append(candidate)
        else:
            result.append(chunk)

    return result


print("Structure-aware split functions ready.")

Structure-aware split functions ready.


In [6]:
def make_chunks_for_document(doc):
    sections = pack_short_sections(doc["sections"])
    output = []
    chunk_index = 0

    for section_index, section in enumerate(sections):
        parts = split_long_text(section["text"])
        parts = add_overlap(parts)

        for part_index, content in enumerate(parts):
            content = content.strip()

            if not content:
                continue

            chunk_index += 1

            chunk_seed = (
                f"{SCHEMA_VERSION}|"
                f"{doc['source']}|"
                f"{doc['document_id']}|"
                f"{section_index}|"
                f"{part_index}|"
                f"{content}"
            )

            chunk_id = hashlib.sha1(
                chunk_seed.encode("utf-8")
            ).hexdigest()[:20]

            chunk = {
                "chunk_id": chunk_id,
                "schema_version": SCHEMA_VERSION,

                "source": doc["source"],
                "document_id": doc["document_id"],
                "original_id": doc["original_id"],

                "chunk_index": chunk_index,
                "section_index": section_index,
                "part_index": part_index,

                "title": doc["title"],
                "section_heading": section["heading"],

                "publication_date": doc["publication_date"],
                "report_period": doc["report_period"],
                "language": doc["language"],
                "url": doc["url"],

                "source_file": doc["source_file"],
                "source_classification": doc["source_classification"],

                "content": content,
                "char_count": len(content),
                "approx_token_count": round(len(content) / 4),
            }

            output.append(chunk)

    return output


chunks = []

for i, doc in enumerate(documents, start=1):
    chunks.extend(make_chunks_for_document(doc))

    if i % 500 == 0:
        print(
            f"Processed {i:,}/{len(documents):,} documents; "
            f"chunks so far={len(chunks):,}"
        )

print("\n=== CHUNKING COMPLETE ===")
print("Documents:", f"{len(documents):,}")
print("Chunks:", f"{len(chunks):,}")

Processed 500/5,058 documents; chunks so far=896
Processed 1,000/5,058 documents; chunks so far=1,644
Processed 1,500/5,058 documents; chunks so far=2,351
Processed 2,000/5,058 documents; chunks so far=3,126
Processed 2,500/5,058 documents; chunks so far=3,955
Processed 3,000/5,058 documents; chunks so far=4,751
Processed 3,500/5,058 documents; chunks so far=5,560
Processed 4,000/5,058 documents; chunks so far=6,312
Processed 4,500/5,058 documents; chunks so far=7,011
Processed 5,000/5,058 documents; chunks so far=17,917

=== CHUNKING COMPLETE ===
Documents: 5,058
Chunks: 19,784


In [7]:
OUTPUT_CHUNKS = OUTPUT_DIR / "chunks_structure_aware_real.jsonl"

with OUTPUT_CHUNKS.open("w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(
            json.dumps(chunk, ensure_ascii=False) + "\n"
        )

print("Saved:", OUTPUT_CHUNKS)
print("Chunk count:", f"{len(chunks):,}")
print("File size (MB):", round(
    OUTPUT_CHUNKS.stat().st_size / (1024 ** 2), 2
))

Saved: D:\Project_Github\gold assisstant\embedding\artifacts\chunking_real\structure_aware\chunks_structure_aware_real.jsonl
Chunk count: 19,784
File size (MB): 44.51


In [8]:
lengths = [c["char_count"] for c in chunks]
chunk_ids = [c["chunk_id"] for c in chunks]

source_document_counts = defaultdict(set)
source_chunk_counts = Counter()

for chunk in chunks:
    source_document_counts[chunk["source"]].add(
        chunk["document_id"]
    )
    source_chunk_counts[chunk["source"]] += 1

sorted_lengths = sorted(lengths)

def percentile(values, q):
    if not values:
        return 0

    index = round((len(values) - 1) * q)
    return values[index]


qa_report = {
    "schema_version": SCHEMA_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "total_documents": len(documents),
    "total_chunks": len(chunks),

    "unique_chunk_ids": len(set(chunk_ids)),
    "duplicate_chunk_ids": len(chunk_ids) - len(set(chunk_ids)),

    "empty_chunks": sum(
        1 for c in chunks if not c["content"].strip()
    ),

    "chunk_char_stats": {
        "min": min(lengths) if lengths else 0,
        "median": statistics.median(lengths) if lengths else 0,
        "mean": round(statistics.mean(lengths), 2) if lengths else 0,
        "p95": percentile(sorted_lengths, 0.95),
        "max": max(lengths) if lengths else 0,
        "over_max_chars": sum(
            1 for n in lengths if n > MAX_CHARS
        ),
    },

    "source_stats": {
        source: {
            "documents": len(source_document_counts[source]),
            "chunks": source_chunk_counts[source],
        }
        for source in INPUTS
    },

    "config": {
        "target_chars": TARGET_CHARS,
        "max_chars": MAX_CHARS,
        "min_chars": MIN_CHARS,
        "overlap_chars": OVERLAP_CHARS,
        "approx_token_count_note":
            "char_count / 4 approximation, not tokenizer-exact",
    },

    "warnings": [
        "Confirm all expected source files were ingested.",
        "Inspect over_max_chars and duplicate IDs before embedding.",
        "Approximate token counts are not exact tokenizer counts.",
        "Source relevance and factual accuracy are not validated by chunking.",
    ],
}

print(json.dumps(qa_report, ensure_ascii=False, indent=2))

{
  "schema_version": "structure_aware_real_v1",
  "created_at_utc": "2026-10-27T07:26:22.254636+00:00",
  "total_documents": 5058,
  "total_chunks": 19784,
  "unique_chunk_ids": 19784,
  "duplicate_chunk_ids": 0,
  "empty_chunks": 0,
  "chunk_char_stats": {
    "min": 1,
    "median": 1199.0,
    "mean": 1108.16,
    "p95": 1781,
    "max": 1800,
    "over_max_chars": 0
  },
  "source_stats": {
    "thai_pbs_gold": {
      "documents": 9,
      "chunks": 128
    },
    "fed_press_releases": {
      "documents": 4553,
      "chunks": 6963
    },
    "bls_cpi": {
      "documents": 248,
      "chunks": 4770
    },
    "bls_nfp": {
      "documents": 248,
      "chunks": 7923
    }
  },
  "config": {
    "target_chars": 1200,
    "max_chars": 1800,
    "min_chars": 250,
    "overlap_chars": 120,
    "approx_token_count_note": "char_count / 4 approximation, not tokenizer-exact"
  },
  "warnings": [
    "Confirm all expected source files were ingested.",
    "Inspect over_max_chars and dup

In [9]:
MANIFEST_PATH = OUTPUT_DIR / "chunking_manifest.json"
ERRORS_PATH = OUTPUT_DIR / "chunking_errors.jsonl"

qa_report["project_root"] = str(PROJECT_ROOT)
qa_report["output_file"] = str(OUTPUT_CHUNKS)
qa_report["input_paths"] = {
    source: [str(path) for path in paths]
    for source, paths in INPUTS.items()
}
qa_report["records_read"] = dict(records_seen)
qa_report["documents_after_dedup_by_source"] = dict(source_counts)
qa_report["ingestion_errors"] = len(errors)

MANIFEST_PATH.write_text(
    json.dumps(qa_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with ERRORS_PATH.open("w", encoding="utf-8") as f:
    for error in errors:
        f.write(json.dumps(error, ensure_ascii=False) + "\n")

print("Manifest:", MANIFEST_PATH)
print("Errors:", ERRORS_PATH)
print("Chunk file:", OUTPUT_CHUNKS)

# สถานะเบื้องต้น ไม่ใช่การรับรองคุณภาพเชิงเนื้อหา
passed = (
    len(chunks) > 0
    and qa_report["empty_chunks"] == 0
    and qa_report["duplicate_chunk_ids"] == 0
)

print("\nQA STATUS:", "PASS - BASIC CHECKS" if passed else "REVIEW REQUIRED")

if qa_report["chunk_char_stats"]["over_max_chars"] > 0:
    print("[WARN] มี Chunk ยาวเกิน MAX_CHARS ให้ตรวจสอบก่อนใช้งาน")

Manifest: D:\Project_Github\gold assisstant\embedding\artifacts\chunking_real\structure_aware\chunking_manifest.json
Errors: D:\Project_Github\gold assisstant\embedding\artifacts\chunking_real\structure_aware\chunking_errors.jsonl
Chunk file: D:\Project_Github\gold assisstant\embedding\artifacts\chunking_real\structure_aware\chunks_structure_aware_real.jsonl

QA STATUS: PASS - BASIC CHECKS


In [10]:
for i, chunk in enumerate(chunks[:5], start=1):
    print("=" * 90)
    print(f"Example {i}")
    print("Source:", chunk["source"])
    print("Document ID:", chunk["document_id"])
    print("Title:", chunk["title"])
    print("Date:", chunk["publication_date"])
    print("Report period:", chunk["report_period"])
    print("Section:", chunk["section_heading"])
    print("Characters:", chunk["char_count"])
    print("-" * 90)
    print(chunk["content"][:1200])
    print()

Example 1
Source: thai_pbs_gold
Document ID: b69cb2366d22735f
Title: สรุปราคาทองคำ 20 ต.ค.2569 ปิดตลาด ผันผวน 21 ครั้ง ลบ 650 บาท
Date: 2026-10-20T17:52:29+07:00
Report period: 
Section: Abstract / Body
Characters: 565
------------------------------------------------------------------------------------------
ราคาทองคำ ลบ 650 บาท ผันผวน 21 ครั้ง ทองลงขณะน้ำมันดิบพุ่ง หลังสหรัฐฯ – อิสราเอล กำลังพิจารณาโจมตีอิหร่าน แต่ดอลลาร์กลับอ่อนค่า "รูปพรรณ" ขายออก 66,400 บาททองคำ "ทองคำแท่ง" ขายออก 65,600 บาททองคำ

วันนี้ (20 ต.ค.2569) "ฮั่วเซ่งเฮง" เปิดเผยภาพรวมราคาทองโลกที่ผ่านมา ราคาทองคำมีแนวโน้มย่อตัวลง จากราคาน้ำมันดิบ Brent และ WTI มีแนวโน้มปรับตัวขึ้นอีกครั้ง หลังจากภาวะการพักรบชั่วคราวและการประวิงเวลาของสหรัฐอเมริกา-อิหร่าน กำลังมีความเสี่ยงที่สูงกว่าเดิม เนื่องจากแรงกดดันจากคำขู่ตัดสายเคเบิลอินเทอร์เน็ตของ IRGC บวกกับการที่เรือพาณิชย์ถูกอิหร่านโจมตีหลายครั้งติดต่อกัน

Example 2
Source: thai_pbs_gold
Document ID: b69cb2366d22735f
Title: สรุปราคาทองคำ 20 ต.ค.2569 ปิดตลาด ผันผวน 21 ครั้ง ลบ 6